In [ ]:
import sys
import importlib
import os

repo_root = os.environ.get("GAP_AWARE_ROOT", os.path.abspath("../../.."))

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

sys.path.insert(0, repo_root)


import model_gap_aware.maskfeat as maskfeat
import model_gap_aware.lmdb_training as lmdb_training

importlib.reload(maskfeat)
importlib.reload(lmdb_training)

LMDBConsecutiveDataModule = lmdb_training.LMDBConsecutiveDataModule
build_model = lmdb_training.build_model

from pytorch_lightning.callbacks import LearningRateMonitor, ModelCheckpoint, TQDMProgressBar
import pytorch_lightning as pl

In [ ]:
# Notebook training config (multi-channel reconstruction)
import os

lmdb_path = os.environ.get(
    "S2_LMDB_PATH",
    "../../../sentinel/data/ssl4eo_consecutive_s2c.lmdb",
)
test_lmdb_path = os.environ.get(
    "S2_TEST_LMDB_PATH",
    lmdb_path.replace(".lmdb", "_test.lmdb"),
)
batch_size = 32
num_workers = 8
val_split = 0.05
prefetch_factor = 8
max_samples = None  # set to an int to cap train+val pool (e.g. 1000 for quick local runs)

# Multi-band target selection (middle frame)
target_channels = [0, 1, 2]

# Input / model shape
image_size = 128
vit_patch_size = 16
num_frames = 5
in_channels = 3
out_channels = len(target_channels)

# Trainer
log_dir = os.path.join(repo_root, "sentinel", "lightning_logs/pretraining")
max_epochs = 2
accelerator = "auto"
devices = "auto"
precision = "16-mixed"

# Build data + model
if not os.path.exists(lmdb_path):
    raise FileNotFoundError(lmdb_path)
if not os.path.exists(test_lmdb_path):
    raise FileNotFoundError(test_lmdb_path)

# Both Sentinel and ESDC LMDBs are now single files (subdir=False)
datamodule = LMDBConsecutiveDataModule(
    lmdb_path=lmdb_path,
    batch_size=batch_size,
    num_workers=num_workers,
    val_split=val_split,
    target_channels=target_channels,
    subdir=False,
    prefetch_factor=prefetch_factor,
    test_lmdb_path=test_lmdb_path,
    max_samples=max_samples,
)

model = build_model(
    image_size=image_size,
    patch_size=vit_patch_size,
    num_frames=num_frames,
    in_channels=in_channels,
    out_channels=out_channels,
)

callbacks = [
    ModelCheckpoint(
        monitor="MAE_val",
        mode="min",
        save_top_k=1,
        filename="gapfiller-{epoch:02d}-{MAE_val:.4f}",
    ),
    LearningRateMonitor(logging_interval="step"),
    # TQDMProgressBar(refresh_rate=50),
]

trainer = pl.Trainer(
    default_root_dir=log_dir,
    max_epochs=max_epochs,
    accelerator=accelerator,
    devices=devices,
    precision=precision,
    callbacks=callbacks,
    log_every_n_steps=10,
)

In [ ]:
trainer.fit(model, datamodule=datamodule)

In [ ]:
trainer.test(model, datamodule=datamodule)